# Add-ons in the notebook

An add-on adds to the editor: a panel under the formula, tools on the
strip, operations, sometimes node types of its own.  In a notebook their
Python runs in **this** kernel, beside the document - the plot's samples
are the kernel's SymPy at work, not a copy of it in the browser.

Four come with sympy-editor.  They are ordinary Python packages, and
anyone can write another (see `addons/template`).


In [ ]:
from pathlib import Path

from sympy import sin, symbols
from sympy_editor import edit, installed_addons, register_addons_folder

x, y = symbols("x y")

# Installed add-ons are found on their own (pip install sympy-editor-plot).
# From a checkout of the repository they are folders, not installed
# packages, so point at them once:
if not installed_addons():
    for folder in (Path("addons"), Path("../addons")):
        if folder.is_dir():
            register_addons_folder(folder)
            break

installed_addons()      # name -> what to load it by


## Switched on from the start

`addons=` opens the editor with them on.  Each may be given by add-on
name (`"plot"`), by module (`"sympy_editor_plot"`), or as an `Addon`
object.

Edit the formula and the graph follows it; select a piece and it plots
the piece.


In [ ]:
edit(sin(x) / x, addons=["plot"])


## Or left for the reader to switch on

`available=` lists them instead, to be turned on and off while editing,
from the top of the drawer the **≡** button opens.  Nothing is loaded
until one is switched on.

This is what a notebook meant for someone else usually wants: the editor
opens plainly, and the tools are there for whoever needs them.

If `installed_addons()` came back empty, none is installed in this
kernel.  Either install them - `pip install sympy-editor-plot` - or, from
a checkout, point at the folder they live in:

```python
from sympy_editor import register_addons_folder
register_addons_folder("addons")      # the repository's own add-ons
```


In [ ]:
edit(x**2 / y - sin(x), available=list(installed_addons()))


## Reading LaTeX, rewriting by rules

Two of the four need a package of their own in the kernel - `lark` for
the LaTeX reader, `sympy-matching` for the rewrite rules.  An add-on
whose Python is missing is listed with the reason rather than silently
absent, so this cell is safe to run either way.

    pip install lark sympy-matching


In [ ]:
w = edit(sin(x) ** 2 + 1, available=list(installed_addons()))
for a in w.document.available_addons():
    print(f"{a['name']:10s} {'on' if a['on'] else 'off':3s} {a.get('error', '')}")
w


## The expression is still yours

Add-ons change what the editor can do, not what it is: `w.expr` is the
live expression whether or not any are on, and `on_change` still fires
on every committed edit.


In [ ]:
w = edit(sin(x) / x, addons=["plot", "tree"])
w.on_change(lambda e: print("now:", e))
w


In [ ]:
w.expr          # whatever the editing has left it as


## Writing one

`addons/template` is a working add-on to copy: a Python package with an
`Addon` in it, and a `static/` folder for the JavaScript and CSS of its
panel.  `addons/README.md` describes the contract.  An add-on installed
in the kernel is found by `installed_addons()` and can be passed here by
name, with no change to sympy-editor itself.
